<div align="center">


## Machine Learning & Text Mining  

## SVM  

</div>

## Application du SVM en Sentiment Analysis

Dans ce TP, nous allons explorer l'analyse de sentiment des critiques de films.  
Nous utiliserons un ensemble de données de critiques de films pré-annotées afin d’entraîner un modèle d’apprentissage automatique capable de prédire si une critique est **positive** ou **négative**.

À partir du dataset partagé sur Google Classroom, il est demandé de développer un modèle intelligent en utilisant des techniques de **Text Mining** et de **Machine Learning**, en suivant les différentes étapes de construction vues en cours.

### Travail à faire :

### 1. Chercher le meilleur modèle SVM en menant une étude empirique sur le type de SVM et les hyperparamètres afin d’avoir le meilleur Score ?

In [52]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC, LinearSVC
from sklearn.metrics import classification_report, accuracy_score , f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
import re
import time
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.feature_extraction.text import TfidfVectorizer



In [53]:
with open("data/reviews.txt", "r", encoding="utf-8") as f:
    reviews = f.read().splitlines()

with open("data/labels.txt", "r", encoding="utf-8") as f:
    labels = f.read().splitlines()


print(len(reviews), len(labels))


df = pd.DataFrame({
    "review": reviews,
    "sentiment": labels
})

df.head()

5000 5000


,review,sentiment
0,bromwell high is a cartoon comedy . it ran at ...,positive
1,story of a man who has unnatural feelings for ...,negative
2,homelessness or houselessness as george carli...,positive
3,airport starts as a brand new luxury pla...,negative
4,brilliant over acting by lesley ann warren . ...,positive


In [54]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"<br\s*/?>", " ", text)   
    text = re.sub(r"[^a-z\s]", " ", text)   
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["review"] = df["review"].apply(clean_text)

In [55]:
X = df["review"]
y = df["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [56]:
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        stop_words="english",
        max_df=0.9,
        min_df=5,
        ngram_range=(1,2),
        sublinear_tf=True
    )),
    ("clf", SVC())
])

In [57]:
param_grid = [
    
    {
        "clf": [LinearSVC()],
        "clf__C": [0.01, 0.1, 1, 10]
    },

  
    {
        "clf": [SVC(kernel="rbf")],
        "clf__C": [0.1, 1, 10],
        "clf__gamma": ["scale", 0.1, 0.01]
    },

   
    {
        "clf": [SVC(kernel="poly")],
        "clf__C": [0.1, 1],
        "clf__degree": [2, 3]
    }
]

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="f1_weighted",
    n_jobs=-1,
    verbose=2
)

grid.fit(X_train, y_train)

Fitting 5 folds for each of 17 candidates, totalling 85 fits
[CV] END .......................clf=LinearSVC(), clf__C=0.01; total time=   1.1s
[CV] END .......................clf=LinearSVC(), clf__C=0.01; total time=   1.1s
[CV] END .......................clf=LinearSVC(), clf__C=0.01; total time=   1.1s
[CV] END .......................clf=LinearSVC(), clf__C=0.01; total time=   1.1s
[CV] END .......................clf=LinearSVC(), clf__C=0.01; total time=   1.1s
[CV] END ........................clf=LinearSVC(), clf__C=0.1; total time=   1.1s
[CV] END ........................clf=LinearSVC(), clf__C=0.1; total time=   1.1s
[CV] END ........................clf=LinearSVC(), clf__C=0.1; total time=   1.1s
[CV] END ........................clf=LinearSVC(), clf__C=0.1; total time=   1.0s
[CV] END ........................clf=LinearSVC(), clf__C=0.1; total time=   1.0s
[CV] END ..........................clf=LinearSVC(), clf__C=1; total time=   0.9s
[CV] END ..........................clf=LinearSVC

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('tfidf',
                                        TfidfVectorizer(max_df=0.9, min_df=5,
                                                        ngram_range=(1, 2),
                                                        stop_words='english',
                                                        sublinear_tf=True)),
                                       ('clf', SVC())]),
             n_jobs=-1,
             param_grid=[{'clf': [LinearSVC()], 'clf__C': [0.01, 0.1, 1, 10]},
                         {'clf': [SVC()], 'clf__C': [0.1, 1, 10],
                          'clf__gamma': ['scale', 0.1, 0.01]},
                         {'clf': [SVC(kernel='poly')], 'clf__C': [0.1, 1],
                          'clf__degree': [2, 3]}],
             scoring='f1_weighted', verbose=2)

In [58]:
print(" Best params:", grid.best_params_)

best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)

print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nReport:\n", classification_report(y_test, y_pred))

 Best params: {'clf': SVC(kernel='poly'), 'clf__C': 1, 'clf__degree': 2}

Accuracy: 0.905

Report:
               precision    recall  f1-score   support

    negative       0.91      0.90      0.90       500
    positive       0.90      0.91      0.91       500

    accuracy                           0.91      1000
   macro avg       0.91      0.91      0.90      1000
weighted avg       0.91      0.91      0.90      1000



### Analyse des Résultats – SVM pour Sentiment Analysis

#### 1. Meilleur Modèle Trouvé

Après une recherche des hyperparamètres (Grid Search), le meilleur modèle obtenu est :

- Modèle : SVM avec noyau polynomial (`poly`)
- Paramètres :
  - C = 1
  - degree = 2

#### 2. Pourquoi ce modèle est le meilleur ?

Le noyau polynomial permet de capturer des relations non linéaires entre les données.

Dans le cas du texte :
- Les sentiments ne sont pas toujours séparables de manière linéaire
- Certaines combinaisons de mots (comme "not good" ou "very bad") nécessitent une modélisation plus complexe

#### Rôle des paramètres

- C = 1  
Ce paramètre contrôle le compromis entre complexité du modèle et généralisation.  
Une valeur intermédiaire comme 1 permet d’éviter à la fois le surapprentissage (overfitting) et le sous-apprentissage (underfitting).

- degree = 2  
Le modèle prend en compte des interactions entre mots de niveau 2.  
Cela permet de capturer des dépendances simples mais importantes sans rendre le modèle trop complexe.

Conclusion :  
Ce modèle est performant car il capture la complexité du langage tout en conservant une bonne capacité de généralisation.

#### 3. Interprétation des performances

##### Accuracy

- Accuracy = 0.905 (~90.5%)

Le modèle classe correctement environ 9 critiques sur 10, ce qui indique une très bonne performance globale.

##### Détail par classe

| Classe   | Precision | Recall | F1-score |
|----------|----------|--------|----------|
| Negative | 0.91     | 0.90   | 0.90     |
| Positive | 0.90     | 0.91   | 0.91     |

##### Precision

- Classe Negative : 0.91  
Lorsque le modèle prédit "negative", il est correct dans 91 % des cas.

- Classe Positive : 0.90  
Lorsque le modèle prédit "positive", il est correct dans 90 % des cas.

La précision est équilibrée entre les deux classes, ce qui montre que le modèle ne favorise pas une classe par rapport à l’autre.

##### Recall

- Classe Negative : 0.90  
Le modèle détecte correctement 90 % des critiques négatives.

- Classe Positive : 0.91  
Le modèle détecte correctement 91 % des critiques positives.

Le recall est également bien équilibré, ce qui signifie que peu d’exemples sont mal classés.

##### F1-score

Le F1-score combine précision et rappel :

- Negative : 0.90  
- Positive : 0.91  

Ces valeurs élevées montrent que le modèle est performant à la fois en précision et en détection.

#### 4. Conclusion

Le modèle SVM avec noyau polynomial (degree = 2, C = 1) offre les meilleures performances sur ce dataset.

Il permet de :
- Capturer des relations non linéaires dans les données textuelles
- Maintenir un bon équilibre entre biais et variance
- Fournir des performances homogènes sur les deux classes

Avec une accuracy de 90.5 % et des F1-scores élevés, ce modèle constitue un excellent choix pour la tâche de classification de sentiments.

### 2. Comparer la performance de SVM avec les autres techniques : Régression, arbre de décision, KNN... et dresser un tableau comparatif en utilisant les mesures de performance que vous jugez pertinentes ?

In [59]:
with open("data/reviews.txt", "r", encoding="utf-8") as f:
    reviews = f.read().splitlines()

with open("data/labels.txt", "r", encoding="utf-8") as f:
    labels = f.read().splitlines()

df = pd.DataFrame({
    "review": reviews,
    "sentiment": labels
})

In [60]:
X_train, X_test, y_train, y_test = train_test_split(
    df["review"], df["sentiment"],
    test_size=0.2,
    random_state=42,
    stratify=df["sentiment"]
)

In [61]:
models = {
    "SVM (Linear)": LinearSVC(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(),
    "KNN": KNeighborsClassifier(n_neighbors=5)
}

results = []



In [62]:
for name, model in models.items():
    
    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(
            stop_words="english",
            max_df=0.9,
            min_df=5,
            ngram_range=(1,2)
        )),
        ("clf", model)
    ])
    
    start = time.time()
    pipeline.fit(X_train, y_train)
    train_time = time.time() - start
    
    y_pred = pipeline.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="weighted")
    
    results.append([name, acc, f1, train_time])

In [63]:
results_df = pd.DataFrame(results, columns=[
    "Model", "Accuracy", "F1-score", "Training Time (s)"
])

results_df = results_df.sort_values(by="F1-score", ascending=False)

print(results_df)

                 Model  Accuracy  F1-score  Training Time (s)
0         SVM (Linear)     0.903  0.902999           0.579024
1  Logistic Regression     0.887  0.886940           0.576765
3                  KNN     0.853  0.852728           0.470372
2        Decision Tree     0.701  0.700976           1.348318


### Analyse Comparative des Modèles

Le tableau des résultats obtenus montre les performances de quatre modèles de classification appliqués à la tâche de sentiment analysis :



### 1. SVM (Linear) – Meilleur modèle

Le modèle SVM linéaire obtient les meilleures performances avec :
- Accuracy ≈ 90.3%
- F1-score ≈ 0.903
- Temps d’entraînement relativement faible

#### Pourquoi il est le meilleur ?

- Les données textuelles vectorisées (TF-IDF) sont **de très haute dimension**  
- Le SVM est spécialement conçu pour gérer ce type de données
- Il maximise la **marge de séparation entre les classes**, ce qui améliore la généralisation
- Il est robuste au bruit et aux données sparsifiées (beaucoup de zéros)

Conclusion :  
Le SVM est particulièrement adapté au NLP, ce qui explique ses meilleures performances.



### 2. Régression Logistique

- Accuracy ≈ 88.7%
- F1-score ≈ 0.887

#### Interprétation

- C’est un modèle linéaire comme SVM
- Il fonctionne bien avec TF-IDF
- Cependant, contrairement au SVM :
  - Il ne maximise pas la marge
  - Il est plus sensible aux données bruitées

Conclusion :  
Performances proches du SVM mais légèrement inférieures car moins robuste.



### 3. K-Nearest Neighbors (KNN)

- Accuracy ≈ 85.3%
- F1-score ≈ 0.853

#### Pourquoi les performances sont plus faibles ?

- KNN repose sur la **distance entre les points**
- En haute dimension (texte), on a le problème de :
  → *curse of dimensionality*
- Les distances deviennent moins significatives
- Le modèle est sensible au bruit

Conclusion :  
KNN n’est pas adapté aux données textuelles vectorisées.



### 4. Arbre de Décision

- Accuracy ≈ 71%
- F1-score ≈ 0.71
- Temps d’entraînement le plus élevé

#### Pourquoi les résultats sont mauvais ?

- Les arbres de décision :
  - Ont tendance à **surapprendre (overfitting)**
  - Ne gèrent pas bien les données de haute dimension
- Le texte TF-IDF contient beaucoup de variables → difficile à partitionner efficacement
- Résultat : mauvaise généralisation

Conclusion :  
Les arbres de décision sont peu adaptés à la classification de texte.



### 5. Analyse du Temps d’Exécution

- **SVM :** rapide et efficace
- **Logistic Regression :** légèrement plus lente
- **KNN :** rapide à entraîner mais coûteux en prédiction
- **Decision Tree :** plus lent ici à cause de la complexité des splits



### 6. Conclusion Générale

Le modèle SVM est le meilleur choix pour cette tâche car :
- Il est adapté aux données de grande dimension
- Il offre le meilleur compromis biais/variance
- Il est robuste et performant

Les autres modèles montrent leurs limites :
- **Logistic Regression :** bonne alternative mais légèrement moins performante
- **KNN :** inadapté aux données textuelles
- **Decision Tree :** faible généralisation

**Conclusion finale :**  
Le SVM est le modèle le plus efficace et le plus fiable pour la classification de sentiments dans ce contexte.